# RAG Pipeline — Notebook de Tests
### Parties couvertes : Indexing · Retrieval · Generation · Evaluation

## 0. Setup & Imports

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

# Vérification de la clé API
api_key = os.getenv("OPENAI_API_KEY")
assert api_key, "❌ OPENAI_API_KEY manquante dans le fichier .env"
print(f"✅ Clé OpenAI chargée : {api_key[:8]}...")

## 1. INDEXING — Chargement et découpage des documents

In [ ]:
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── Changer ce chemin par votre PDF de test ──────────────────────────────────
PDF_PATH = "../assets/sample.pdf"

# Extraction du texte brut
reader = PdfReader(PDF_PATH)
raw_text = ""
for page in reader.pages:
    raw_text += page.extract_text() or ""

print(f"📄 Nombre de pages  : {len(reader.pages)}")
print(f"📝 Caractères total : {len(raw_text):,}")
print("\n── Extrait (500 premiers caractères) ──")
print(raw_text[:500])

In [ ]:
# Découpage (chunking)
splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=512,
    chunk_overlap=50,
)
chunks = splitter.split_text(raw_text)

print(f"✂️  Nombre de chunks : {len(chunks)}")
print(f"📏 Taille moyenne   : {sum(len(c) for c in chunks) // len(chunks)} caractères")
print("\n── Exemple de chunk ──")
print(chunks[0])

## 2. INDEXING — Vectorisation et stockage dans ChromaDB

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

# Modèle d'embedding OpenAI
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

# Création du vector store (persist_directory = stockage sur disque)
vector_store = Chroma.from_texts(
    texts=chunks,
    embedding=embedding_model,
    collection_name="rag_collection",
    persist_directory="../chroma_db",
)

print(f"✅ Index créé : {vector_store._collection.count()} vecteurs stockés")

## 3. RETRIEVAL — Recherche de documents pertinents

In [ ]:
# Créer le retriever (k = nombre de chunks retournés)
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

# Question de test
question = "De quoi parle ce document ?"
docs = retriever.invoke(question)

print(f"🔍 Question : {question}")
print(f"📦 {len(docs)} chunks récupérés :\n")
for i, doc in enumerate(docs):
    print(f"--- Chunk {i+1} ---")
    print(doc.page_content[:300])
    print()

In [ ]:
# Recherche avec score de similarité
results_with_scores = vector_store.similarity_search_with_score(question, k=4)

print("🎯 Résultats avec scores de similarité (distance L2, plus bas = plus proche) :\n")
for doc, score in results_with_scores:
    print(f"  Score : {score:.4f} | {doc.page_content[:150]}...")

## 4. GENERATION — Réponse avec le LLM (GPT-4o)

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# LLM
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# Prompt template
prompt = ChatPromptTemplate.from_template("""
Réponds à la question suivante en te basant UNIQUEMENT sur le contexte fourni.
Si la réponse n'est pas dans le contexte, dis "Je ne sais pas".

<context>
{context}
</context>

Question : {question}
Réponse :
""")

# Fonction utilitaire pour formater les docs
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Chaîne RAG LCEL (LangChain Expression Language)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Test
response = rag_chain.invoke(question)
print(f"❓ Question : {question}")
print(f"\n💬 Réponse :\n{response}")

In [ ]:
# Test avec streaming
print("🔄 Réponse en streaming :\n")
for chunk in rag_chain.stream(question):
    print(chunk, end="", flush=True)
print()

## 5. EVALUATION — Métriques RAG avec RAGAS

In [ ]:
# Jeu de données d'évaluation (questions + réponses de référence)
eval_questions = [
    "De quoi parle ce document ?",
    "Quels sont les points principaux ?",
    "Quelle est la conclusion ?",
]

# Génération des réponses + contextes pour évaluation
eval_data = []
for q in eval_questions:
    ctx_docs = retriever.invoke(q)
    ctx_texts = [d.page_content for d in ctx_docs]
    answer = rag_chain.invoke(q)
    eval_data.append({
        "question": q,
        "answer": answer,
        "contexts": ctx_texts,
    })
    print(f"✅ {q[:50]}...")

print(f"\n{len(eval_data)} exemples générés pour l'évaluation")

In [ ]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,          # La réponse est-elle fidèle au contexte ?
    answer_relevancy,      # La réponse est-elle pertinente par rapport à la question ?
    context_precision,     # Les chunks récupérés sont-ils précis ?
    context_recall,        # Tous les éléments nécessaires sont-ils dans le contexte ?
)
from datasets import Dataset

# Création du dataset RAGAS
ragas_dataset = Dataset.from_list(eval_data)

# Évaluation
results = evaluate(
    dataset=ragas_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision],
)

print("\n📊 Résultats d'évaluation RAGAS :")
print(results)

In [ ]:
# Affichage sous forme de DataFrame
import pandas as pd

df = results.to_pandas()
print(df[["question", "faithfulness", "answer_relevancy", "context_precision"]].to_string())